# Gravitino + Daft 探索 Notebook

本 notebook 用于在本地 Gravitino Playground 中探索 GVFS 读写能力。

前置条件：
- Gravitino Playground 已启动（http://localhost:8090）
- 已运行 `scripts/bootstrap_gravitino.py` 创建 catalog/schema/filesets
- 已运行 `scripts/upload_sample_data.py` 上传示例数据

In [ ]:
import sys
sys.path.insert(0, '..')

from gravitino_daft.config import gravitino_io_config, gvfs_path, INPUT_FILESET, OUTPUT_FILESET
import daft

## 1. 读取 GVFS fileset 中的 Parquet

In [ ]:
io_config = gravitino_io_config()
df = daft.read_parquet(gvfs_path(INPUT_FILESET, '*.parquet'), io_config=io_config)
df.show()

## 2. 写入 GVFS fileset

In [ ]:
sample = daft.from_pydict({
    'id': [1, 2, 3],
    'label': ['x', 'y', 'z'],
    'value': [1.1, 2.2, 3.3],
})
sample.write_parquet(gvfs_path(OUTPUT_FILESET, 'notebook_sample.parquet'), io_config=io_config)

read_back = daft.read_parquet(gvfs_path(OUTPUT_FILESET, 'notebook_sample.parquet'), io_config=io_config)
read_back.show()

## 3. 使用 Gravitino Python client 列出 filesets

In [ ]:
from gravitino import GravitinoClient

client = GravitinoClient(
    uri='http://localhost:8090',
    metalake_name='metalake_demo',
)
filesets = client.list_filesets('demo_fileset_catalog', 'demo')
print(filesets)